In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
sys.path.insert(0,'..')
from scripts.utils import Config
from scripts.models.sr.train import load_kernels
from scripts.models.sr.equations import load_registry,calc_physical_constants,calc_physical_precip
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR   = CONFIGS['filepaths']['splits']
MODELSDIR   = CONFIGS['filepaths']['models']
WEIGHTSFROM = CONFIGS['experiments']['sr']['runs']['sr_atm']['weightsfrom']
FIELDVARS   = CONFIGS['experiments']['nn']['runs'][WEIGHTSFROM]['fieldvars']
SPLIT       = 'test'
MINSAMPLES  = 50
MINPRECIP   = 0.1
LANDFRAC    = 0.5
MIBINS      = 35
FLUXBINS    = 20
MASKCOLOR   = '0.75'
FIGPATH     = '../figs/fig_maskedbins.jpg'
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
REGISTRY    = load_registry(MODELSDIR)

In [ ]:
def bin_mean(x,y,values,nbins):
    edges   = (np.linspace(*np.percentile(x,[1,99]),nbins+1),np.linspace(*np.percentile(y,[1,99]),nbins+1))
    binidx  = np.clip(np.digitize(x,edges[0])-1,0,nbins-1)*nbins+np.clip(np.digitize(y,edges[1])-1,0,nbins-1)
    counts  = np.bincount(binidx,minlength=nbins**2)
    means   = np.where(counts>=MINSAMPLES,np.bincount(binidx,weights=values,minlength=nbins**2)/np.maximum(counts,1),np.nan).reshape(nbins,nbins).T
    centers = [0.5*(edge[:-1]+edge[1:]) for edge in edges]
    return centers,edges,means,binidx,counts

def summarize(label,binidx,counts,tp):
    masked = counts<MINSAMPLES
    insamp = masked[binidx]
    print(f'{label}: {int(masked.sum())}/{counts.size} bins masked ({int((masked&(counts>0)).sum())} non-empty) | '
          f'{insamp.mean():.3%} of samples | {tp[insamp].sum()/tp.sum():.3%} of ERA5 precipitation')

In [ ]:
atmphysical = calc_physical_constants('sr_atm_eq',REGISTRY,STATS)
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    dsig   = ds['dsig'].values
    fields = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,dsig.size) for name in FIELDVARS],axis=1)
    tp,shf,lhf = (ds[name].transpose('time','lat','lon').values.ravel() for name in ['tp','shf','lhf'])
    lf     = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel().astype(np.float64)
kernels = load_kernels(Config(),WEIGHTSFROM,dsig)

integrals   = {var:fields[:,i,:]@(kernels[var]*dsig) for i,var in enumerate(FIELDVARS)}
inputs      = {**integrals,'lf':lf,'shf':shf,'lhf':lhf}
moisture    = atmphysical['kappa']*(integrals['rh']-STATS['rh_mean'])
instability = integrals['thetae']-atmphysical['gamma']*integrals['thetaestar']-atmphysical['thetac']
thetaeanom  = integrals['thetae']-STATS['thetae_mean']
atmtp       = calc_physical_precip('sr_atm_eq',atmphysical,inputs,STATS)

finite    = np.isfinite(moisture)&np.isfinite(instability)&np.isfinite(tp)
sfcfinite = finite&np.isfinite(atmtp)&np.isfinite(shf)&np.isfinite(lhf)&np.isfinite(lf)
oceanmask = sfcfinite&(atmtp>=MINPRECIP)&(lf<LANDFRAC)
panels = {
    'Figure 4b':(moisture[finite],instability[finite],tp[finite],tp[finite],MIBINS),
    'Figure 5d':(shf[oceanmask],lhf[oceanmask],(tp-atmtp)[oceanmask],tp[oceanmask],FLUXBINS),
    'Figure 6d':(shf[oceanmask],thetaeanom[oceanmask],(tp-atmtp)[oceanmask],tp[oceanmask],FLUXBINS)}
binned = {}
for label,(x,y,values,weights,nbins) in panels.items():
    centers,edges,means,binidx,counts = bin_mean(x,y,values,nbins)
    summarize(label,binidx,counts,weights)
    binned[label] = (centers,edges,means)
print(f'Figure 5d/6d samples (ocean, SR-ATM >= {MINPRECIP} mm): {oceanmask.sum():,} of {sfcfinite.sum():,}')

In [ ]:
fig,axs = pplt.subplots(ncols=3,refwidth=2.2,share=False,wspace=1.5)
for ax,(label,(centers,edges,means)) in zip(axs,binned.items()):
    ax.set_facecolor(MASKCOLOR)
    ax.set_box_aspect(1)
    ax.format(title=label,xlim=(edges[0][0],edges[0][-1]),ylim=(edges[1][0],edges[1][-1]))
m0 = axs[0].pcolormesh(*binned['Figure 4b'][0],binned['Figure 4b'][2],cmap='ColdHot_r',cmap_kw={'left':0.52},vmin=0,vmax=4,levels=18,extend='max')
m1 = axs[1].pcolormesh(*binned['Figure 5d'][0],binned['Figure 5d'][2],cmap='ColdHot_r',vmin=-4,vmax=4,levels=32,extend='both')
axs[2].pcolormesh(*binned['Figure 6d'][0],binned['Figure 6d'][2],cmap='ColdHot_r',vmin=-4,vmax=4,levels=32,extend='both')
diagonal = [min(binned['Figure 4b'][1][0][0],binned['Figure 4b'][1][1][0]),max(binned['Figure 4b'][1][0][-1],binned['Figure 4b'][1][1][-1])]
axs[0].plot(diagonal,diagonal,'k--',linewidth=1)
axs[0].format(xlabel='$\\mathit{M}$ (K)',ylabel='$\\mathit{I}$ (K)')
axs[1].format(xlabel='SHF (W/m$^2$)',ylabel='LHF (W/m$^2$)')
axs[2].format(xlabel='SHF (W/m$^2$)',ylabel='$\\widehat{\\mathit{\\theta_e}}-\\mathit{\\mu_{\\theta_e}}$ (K)')
axs[0].colorbar(m0,loc='b',label='Total Precipitation (mm)')
fig.colorbar(m1,loc='b',cols=(2,3),ticks=1,label='$\\Delta$Precipitation (mm)')
fig.format(abc=True,titleloc='l',grid=False)
pplt.show()
fig.save(FIGPATH)